## Model Evaluation and Interpretation

### Objective
This notebook evaluates the Cox Proportional Hazards model built on the METABRIC clinical dataset
and interprets the estimated clinical risk factors.

Main goals:
- review model outputs
- identify significant variables
- interpret hazard ratios with confidence intervals
- summarize clinical implications
- document limitations and future directions

---

## 목적
이 노트북의 목적은 METABRIC 임상 데이터 기반 Cox Proportional Hazards 모델의 결과를 평가하고, 추정된 위험인자의 임상적 의미를 해석하는 것이다.

핵심 목표:
- 모델 결과 검토
- 유의한 변수 확인
- hazard ratio 및 신뢰구간 해석
- 임상적 의미 요약
- 한계점 및 향후 방향 정리

In [14]:
# 1. Import
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

In [15]:
# 2. Load model outputs
summary_path = "../data/processed/coxph_model_summary.csv"
hr_path = "../data/processed/coxph_hazard_ratios.csv"

summary_df = pd.read_csv(summary_path)
hr_df = pd.read_csv(hr_path)

print("Summary shape:", summary_df.shape)
print("HR table shape:", hr_df.shape)

display(summary_df.head())
display(hr_df.head())

Summary shape: (10, 12)
HR table shape: (10, 9)


,covariate,coef,exp(coef),se(coef),coef lower 95%,coef upper 95%,exp(coef) lower 95%,exp(coef) upper 95%,cmp to,z,p,-log2(p)
0,Age at Diagnosis,0.0054,1.0055,0.0040,-0.0024,0.0132,0.9976,1.0133,0.0000,1.3677,0.1714,2.5446
1,Lymph nodes examined positive,0.0577,1.0594,0.0108,0.0366,0.0788,1.0373,1.0820,0.0000,5.3596,0.0000,23.5151
2,Tumor Size,0.0088,1.0088,0.0027,0.0035,0.0141,1.0035,1.0142,0.0000,3.2287,0.0012,9.6513
3,Neoplasm Histologic Grade_2.0,0.3644,1.4396,0.2393,-0.1047,0.8334,0.9006,2.3012,0.0000,1.5225,0.1279,2.9672
4,Neoplasm Histologic Grade_3.0,0.5939,1.8110,0.2381,0.1273,1.0605,1.1357,2.8878,0.0000,2.4946,0.0126,6.3091


,covariate,coef,exp(coef),exp(coef) lower 95%,exp(coef) upper 95%,se(coef),z,p,-log2(p)
0,Lymph nodes examined positive,0.0577,1.0594,1.0373,1.0820,0.0108,5.3596,0.0000,23.5151
1,HER2 Status_Positive,0.4552,1.5765,1.2222,2.0336,0.1299,3.5046,0.0005,11.0948
2,Tumor Stage_2.0,0.3981,1.4890,1.1767,1.8842,0.1201,3.3146,0.0009,10.0897
3,Tumor Size,0.0088,1.0088,1.0035,1.0142,0.0027,3.2287,0.0012,9.6513
4,Tumor Stage_3.0,0.6132,1.8463,1.2237,2.7854,0.2098,2.9223,0.0035,8.1687


In [16]:
# 3. Standardize feature column names
def standardize_feature_column(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "Unnamed: 0" in df.columns:
        df = df.rename(columns={"Unnamed: 0": "feature"})
    elif df.columns[0] not in [
        "feature", "coef", "exp(coef)", "se(coef)", "z", "p",
        "exp(coef) lower 95%", "exp(coef) upper 95%"
    ]:
        df = df.rename(columns={df.columns[0]: "feature"})
    return df

summary_df = standardize_feature_column(summary_df)
hr_df = standardize_feature_column(hr_df)

print("Summary columns:")
print(summary_df.columns.tolist())

print("\nHR columns:")
print(hr_df.columns.tolist())

Summary columns:
['feature', 'coef', 'exp(coef)', 'se(coef)', 'coef lower 95%', 'coef upper 95%', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'cmp to', 'z', 'p', '-log2(p)']

HR columns:
['feature', 'coef', 'exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'se(coef)', 'z', 'p', '-log2(p)']


In [17]:
# 4. Ensure required columns exist
required_hr_cols = [
    "feature",
    "coef",
    "exp(coef)",
    "exp(coef) lower 95%",
    "exp(coef) upper 95%",
    "p"
]

missing_cols = [col for col in required_hr_cols if col not in hr_df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns in hr_df: {missing_cols}")

print("All required HR columns are available.")

All required HR columns are available.


In [18]:
# 5. Review full hazard ratio table
hr_df = hr_df.sort_values(by="p").reset_index(drop=True)
hr_df

,feature,coef,exp(coef),exp(coef) lower 95%,exp(coef) upper 95%,se(coef),z,p,-log2(p)
0,Lymph nodes examined positive,0.0577,1.0594,1.0373,1.0820,0.0108,5.3596,0.0000,23.5151
1,HER2 Status_Positive,0.4552,1.5765,1.2222,2.0336,0.1299,3.5046,0.0005,11.0948
2,Tumor Stage_2.0,0.3981,1.4890,1.1767,1.8842,0.1201,3.3146,0.0009,10.0897
3,Tumor Size,0.0088,1.0088,1.0035,1.0142,0.0027,3.2287,0.0012,9.6513
4,Tumor Stage_3.0,0.6132,1.8463,1.2237,2.7854,0.2098,2.9223,0.0035,8.1687
5,Tumor Stage_4.0,1.1942,3.3011,1.4529,7.5005,0.4187,2.8520,0.0043,7.8466
6,Neoplasm Histologic Grade_3.0,0.5939,1.8110,1.1357,2.8878,0.2381,2.4946,0.0126,6.3091
7,ER Status_Positive,-0.2523,0.7770,0.6160,0.9801,0.1185,-2.1292,0.0332,4.9111
8,Neoplasm Histologic Grade_2.0,0.3644,1.4396,0.9006,2.3012,0.2393,1.5225,0.1279,2.9672
9,Age at Diagnosis,0.0054,1.0055,0.9976,1.0133,0.0040,1.3677,0.1714,2.5446


In [19]:
# 6. Significant and non-significant variables
significant_df = hr_df[hr_df["p"] < 0.05].copy().sort_values(by="p")
nonsignificant_df = hr_df[hr_df["p"] >= 0.05].copy().sort_values(by="p")

print("Number of significant features:", significant_df.shape[0])
print("Number of non-significant features:", nonsignificant_df.shape[0])

display(significant_df)
display(nonsignificant_df)

Number of significant features: 8
Number of non-significant features: 2


,feature,coef,exp(coef),exp(coef) lower 95%,exp(coef) upper 95%,se(coef),z,p,-log2(p)
0,Lymph nodes examined positive,0.0577,1.0594,1.0373,1.0820,0.0108,5.3596,0.0000,23.5151
1,HER2 Status_Positive,0.4552,1.5765,1.2222,2.0336,0.1299,3.5046,0.0005,11.0948
2,Tumor Stage_2.0,0.3981,1.4890,1.1767,1.8842,0.1201,3.3146,0.0009,10.0897
3,Tumor Size,0.0088,1.0088,1.0035,1.0142,0.0027,3.2287,0.0012,9.6513
4,Tumor Stage_3.0,0.6132,1.8463,1.2237,2.7854,0.2098,2.9223,0.0035,8.1687
5,Tumor Stage_4.0,1.1942,3.3011,1.4529,7.5005,0.4187,2.8520,0.0043,7.8466
6,Neoplasm Histologic Grade_3.0,0.5939,1.8110,1.1357,2.8878,0.2381,2.4946,0.0126,6.3091
7,ER Status_Positive,-0.2523,0.7770,0.6160,0.9801,0.1185,-2.1292,0.0332,4.9111


,feature,coef,exp(coef),exp(coef) lower 95%,exp(coef) upper 95%,se(coef),z,p,-log2(p)
8,Neoplasm Histologic Grade_2.0,0.3644,1.4396,0.9006,2.3012,0.2393,1.5225,0.1279,2.9672
9,Age at Diagnosis,0.0054,1.0055,0.9976,1.0133,0.0040,1.3677,0.1714,2.5446


In [20]:
# 7. Risk-increasing and protective features
risk_increasing_df = significant_df[significant_df["exp(coef)"] > 1].copy()
risk_increasing_df = risk_increasing_df.sort_values(by="exp(coef)", ascending=False)

protective_df = significant_df[significant_df["exp(coef)"] < 1].copy()
protective_df = protective_df.sort_values(by="exp(coef)", ascending=True)

print("Risk-increasing features:", risk_increasing_df.shape[0])
print("Protective features:", protective_df.shape[0])

display(risk_increasing_df)
display(protective_df)

Risk-increasing features: 7
Protective features: 1


,feature,coef,exp(coef),exp(coef) lower 95%,exp(coef) upper 95%,se(coef),z,p,-log2(p)
5,Tumor Stage_4.0,1.1942,3.3011,1.4529,7.5005,0.4187,2.8520,0.0043,7.8466
4,Tumor Stage_3.0,0.6132,1.8463,1.2237,2.7854,0.2098,2.9223,0.0035,8.1687
6,Neoplasm Histologic Grade_3.0,0.5939,1.8110,1.1357,2.8878,0.2381,2.4946,0.0126,6.3091
1,HER2 Status_Positive,0.4552,1.5765,1.2222,2.0336,0.1299,3.5046,0.0005,11.0948
2,Tumor Stage_2.0,0.3981,1.4890,1.1767,1.8842,0.1201,3.3146,0.0009,10.0897
0,Lymph nodes examined positive,0.0577,1.0594,1.0373,1.0820,0.0108,5.3596,0.0000,23.5151
3,Tumor Size,0.0088,1.0088,1.0035,1.0142,0.0027,3.2287,0.0012,9.6513


,feature,coef,exp(coef),exp(coef) lower 95%,exp(coef) upper 95%,se(coef),z,p,-log2(p)
7,ER Status_Positive,-0.2523,0.7770,0.6160,0.9801,0.1185,-2.1292,0.0332,4.9111


In [21]:
# 8. Build evaluation table
evaluation_table = significant_df[[
    "feature",
    "coef",
    "exp(coef)",
    "exp(coef) lower 95%",
    "exp(coef) upper 95%",
    "p"
]].copy()

evaluation_table = evaluation_table.sort_values(by="p").reset_index(drop=True)

print("Evaluation table shape:", evaluation_table.shape)
evaluation_table

Evaluation table shape: (8, 6)


,feature,coef,exp(coef),exp(coef) lower 95%,exp(coef) upper 95%,p
0,Lymph nodes examined positive,0.0577,1.0594,1.0373,1.0820,0.0000
1,HER2 Status_Positive,0.4552,1.5765,1.2222,2.0336,0.0005
2,Tumor Stage_2.0,0.3981,1.4890,1.1767,1.8842,0.0009
3,Tumor Size,0.0088,1.0088,1.0035,1.0142,0.0012
4,Tumor Stage_3.0,0.6132,1.8463,1.2237,2.7854,0.0035
5,Tumor Stage_4.0,1.1942,3.3011,1.4529,7.5005,0.0043
6,Neoplasm Histologic Grade_3.0,0.5939,1.8110,1.1357,2.8878,0.0126
7,ER Status_Positive,-0.2523,0.7770,0.6160,0.9801,0.0332


In [22]:
# 9. Add interpretation labels
def interpret_hr(hr: float) -> str:
    if hr > 1:
        return "Risk Increasing"
    elif hr < 1:
        return "Protective"
    else:
        return "Neutral"


def effect_size_label(hr: float) -> str:
    if hr >= 2.0:
        return "Strong increase"
    elif hr > 1.2:
        return "Moderate increase"
    elif hr > 1.0:
        return "Mild increase"
    elif hr <= 0.5:
        return "Strong protection"
    elif hr < 0.8:
        return "Moderate protection"
    elif hr < 1.0:
        return "Mild protection"
    else:
        return "Neutral"

evaluation_table["effect_direction"] = evaluation_table["exp(coef)"].apply(interpret_hr)
evaluation_table["effect_size"] = evaluation_table["exp(coef)"].apply(effect_size_label)

evaluation_table

,feature,coef,exp(coef),exp(coef) lower 95%,exp(coef) upper 95%,p,effect_direction,effect_size
0,Lymph nodes examined positive,0.0577,1.0594,1.0373,1.0820,0.0000,Risk Increasing,Mild increase
1,HER2 Status_Positive,0.4552,1.5765,1.2222,2.0336,0.0005,Risk Increasing,Moderate increase
2,Tumor Stage_2.0,0.3981,1.4890,1.1767,1.8842,0.0009,Risk Increasing,Moderate increase
3,Tumor Size,0.0088,1.0088,1.0035,1.0142,0.0012,Risk Increasing,Mild increase
4,Tumor Stage_3.0,0.6132,1.8463,1.2237,2.7854,0.0035,Risk Increasing,Moderate increase
5,Tumor Stage_4.0,1.1942,3.3011,1.4529,7.5005,0.0043,Risk Increasing,Strong increase
6,Neoplasm Histologic Grade_3.0,0.5939,1.8110,1.1357,2.8878,0.0126,Risk Increasing,Moderate increase
7,ER Status_Positive,-0.2523,0.7770,0.6160,0.9801,0.0332,Protective,Moderate protection


In [23]:
# 10. Final interpretation table
final_interpretation_table = evaluation_table[[
    "feature",
    "exp(coef)",
    "exp(coef) lower 95%",
    "exp(coef) upper 95%",
    "p",
    "effect_direction",
    "effect_size"
]].copy()

final_interpretation_table = final_interpretation_table.sort_values(by="p").reset_index(drop=True)

print("Final interpretation table shape:", final_interpretation_table.shape)
final_interpretation_table

Final interpretation table shape: (8, 7)


,feature,exp(coef),exp(coef) lower 95%,exp(coef) upper 95%,p,effect_direction,effect_size
0,Lymph nodes examined positive,1.0594,1.0373,1.0820,0.0000,Risk Increasing,Mild increase
1,HER2 Status_Positive,1.5765,1.2222,2.0336,0.0005,Risk Increasing,Moderate increase
2,Tumor Stage_2.0,1.4890,1.1767,1.8842,0.0009,Risk Increasing,Moderate increase
3,Tumor Size,1.0088,1.0035,1.0142,0.0012,Risk Increasing,Mild increase
4,Tumor Stage_3.0,1.8463,1.2237,2.7854,0.0035,Risk Increasing,Moderate increase
5,Tumor Stage_4.0,3.3011,1.4529,7.5005,0.0043,Risk Increasing,Strong increase
6,Neoplasm Histologic Grade_3.0,1.8110,1.1357,2.8878,0.0126,Risk Increasing,Moderate increase
7,ER Status_Positive,0.7770,0.6160,0.9801,0.0332,Protective,Moderate protection


In [24]:
# 11. Save evaluation outputs
output_path = "../data/processed/coxph_evaluation_summary.csv"
final_interpretation_table.to_csv(output_path, index=False)

print(f"Saved evaluation summary to: {output_path}")

Saved evaluation summary to: ../data/processed/coxph_evaluation_summary.csv


## Interpretation Guide

A hazard ratio (HR) greater than 1 indicates increased survival risk,
while an HR less than 1 indicates a protective effect.

Examples:
- HR = 1.50 → 50% higher hazard
- HR = 0.80 → 20% lower hazard

If the 95% confidence interval does not cross 1,
the effect is more likely to be statistically meaningful.

---

## 해석 기준

hazard ratio(HR)가 1보다 크면 위험 증가, 1보다 작으면 보호 효과를 의미한다.

예시:
- HR = 1.50 → 위험 50% 증가
- HR = 0.80 → 위험 20% 감소

95% 신뢰구간이 1을 지나지 않으면
통계적으로 더 의미 있는 효과일 가능성이 높다.

## Key Findings

Based on the Cox PH model:

- Higher tumor stage, higher histologic grade, HER2 positivity,
  larger tumor size, and increased lymph node positivity are associated with increased hazard.
- ER positive status appears to be associated with lower hazard.
- Variables from very small subgroups should be interpreted cautiously.

---

## 핵심 결과

Cox PH 모델 결과를 바탕으로 다음과 같은 해석이 가능하다.

- 높은 종양 병기, 높은 조직학적 grade, HER2 양성,
  큰 종양 크기, 많은 양성 림프절 수는 위험 증가와 관련될 가능성이 높다.
- ER 양성 상태는 상대적으로 낮은 위험과 관련될 수 있다.
- 매우 작은 subgroup에서 추정된 변수는 해석에 주의가 필요하다.

## Clinical Interpretation

The model suggests that disease progression-related variables
such as tumor stage, tumor grade, lymph node positivity, and HER2 status
are associated with increased mortality risk.

ER positivity appears to be associated with lower hazard,
which is clinically plausible in breast cancer prognosis.

---

## 임상적 해석

모델 결과에 따르면 종양 병기, 종양 grade, 양성 림프절 수, HER2 상태와 같은 질병 진행 관련 변수들은 사망 위험 증가와 연관될 가능성이 있다.

반면 ER 양성은 상대적으로 낮은 hazard와 연결되어,
유방암 예후 측면에서 임상적으로 타당한 방향성을 보인다.

## Limitations

This evaluation has several limitations:

- The model was reviewed on the same processed dataset used for fitting.
- External validation was not performed.
- Some subgroup categories may contain very small numbers of samples.
- This is an interpretable baseline Cox PH model, not a fully optimized survival model.

---

## 한계점

이번 평가는 다음과 같은 한계가 있다.

- 모델 적합에 사용한 동일 데이터셋에서 결과를 검토하였다.
- 외부 검증은 수행하지 않았다.
- 일부 subgroup 범주는 표본 수가 매우 적을 수 있다.
- 본 모델은 해석 가능한 baseline Cox PH 모델이며, 최종 최적화 모델은 아니다.

## Future Improvements

Possible next steps include:

- internal validation with resampling
- train / validation split for survival modeling
- regularized Cox models
- comparison with Random Survival Forest or DeepSurv
- more robust subgroup handling

---

## 향후 개선 방향

다음과 같은 방향으로 확장할 수 있다.

- resampling 기반 내부 검증
- survival modeling용 train / validation 분리
- regularized Cox model 적용
- Random Survival Forest, DeepSurv와의 비교
- subgroup 처리의 안정성 강화

## Final Conclusion

The baseline Cox PH model successfully identified clinically interpretable variables
associated with breast cancer survival risk in the METABRIC dataset.

This evaluation confirms that the current pipeline functions
as a reproducible and interpretable survival analysis portfolio project.

---

## 최종 결론

Baseline Cox PH 모델은 METABRIC 데이터에서 유방암 생존 위험과 관련된 임상적으로 해석 가능한 변수들을 식별하였다.

이번 평가는 현재 파이프라인이
재현 가능하고 해석 가능한 survival analysis 포트폴리오 프로젝트로 기능함을 보여준다.